# Scenario 2: <sup>31</sup>P-NMR kinetic analysis of an enzymatic cascade reaction

## Data Processing and Analysis Notebook

---

### Notebook Setup

The essential packages to work with NMRPy and the EnzymeML extension are:

- `matplotlib`
- `nmrpy`
- `pyenzyme`

Depending on the use case, some additional packages maybe useful:

- `numpy`
- `pandas`
- `scipy`
- `pickle`
- `...`

Interactive plotting requires either the `TkAgg` or `ipympl` backend. While the `ipympl` backend allows for convenience interactive inline plotting in the Jupyter notebook, it does not run as smoothly as the `TkAgg` backend that pops up the graphs in separate windows. If interactivity is not required, `matplotlib.use("ipympl")` can be commented out and uncommenting `%matplotlib inline` will plot static graphs inline instead.

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
import numpy as np

import nmrpy
import pyenzyme as pe

plt.ioff()
matplotlib.use('ipympl')
# %matplotlib inline

---

### Processing of the NMR Data

#### NMR data loading

An NMRPy workflow starts with the loading of the NMR data. The NMRPy package supports a variety of file formats, including Bruker, Varian, and more. When an EnzymeML file is used, like in this scenario, it can be loaded into NMRPy using the `parse_enzymeml_document` function, which extracts the species and measurement metadata and links it with the internal NMRPy data model.

In [ ]:
p = nmrpy.from_path('./data/input/nmr_10mmPEP_10lys_stdpH7_scerev_190821')

#### EnzymeML document loading

In [ ]:
p.parse_enzymeml_document('./data/input/pgm-eno.json')
CURRENT_MEASUREMENT = p.enzymeml_document.measurements[-1].id

#### Apodization, zero-filling, and Fourier transformation

It is common to improve the signal-to-noise ratio of NMR spectra by applying apodization. Also, the data has to be transformed into the frequency domain by Fourier transformation.

In [ ]:
p.emhz_fids(lb=5)       # Apodization
p.zf_fids()             # Zero-filling
p.ft_fids()             # Fourier transformation

In [ ]:
p.plot_array()

#### Phase correction, normalization, and realization

Oftentimes, the spectra need to be phase corrected, which can either be done automatically, using the `phase_correct_fids` method, or manually using the `phaser` widget. At this stage, the spectra can be realized, which discards the imaginary part of the spectra, and normalized by the maximum data value among all spectra.

In [ ]:
p.phase_correct_fids()  # Phase correction
p.real_fids()           # Discard imaginary part
p.norm_fids()           # Normalization

In [ ]:
p.plot_array()

#### Calibration

The spectra likely need to be calibrated by setting the ppm of a reference peak, which can be done using the `calibrate` method.

In [ ]:
p.calibrate()           # 0.44 ppm for TEP

#### Baseline correction

In case of a noisy baseline, it can be corrected using the `baseliner_fids` and `baseline_correct_fids` methods for setting and applying a baseline correction.

In [ ]:
p.baseliner_fids()

In [ ]:
p.baseline_correct_fids()

#### Peak picking

The peak assignment can be done using the `Fid.peakpicker` method or by manually setting the `Fid.peaks` and `Fid.ranges` attributes.

In [ ]:
p.peakpicker(voff=1e5)

#### Deconvolution

After picking the relevant peaks, these can be deconvoluted, making their integrals available for the subsequent analysis. There are also special plotting methods for visualizing the deconvolution results, such as `plot_deconv_array`.

In [ ]:
p.deconv_fids()         # Deconvolution

In [ ]:
p.plot_deconv_array(
    upper_ppm=3.5,
    lower_ppm=-2,
    residual_colour=None,
    azim=-95,
    elev=20
)

#### Peak assignment

Species may be assigned to peaks either by using an EnzymeML document (the default) if available or by providing a list of species to the respective method through the `species_list` argument.

When assigning species on the level of the entire FID array, an `index_list` of the FIDs to assign can be provided, effectively allowing to work on a slice of the FID array.

In [ ]:
p.assign_peaks()

---

### Analysis of the enzymatic cascade reaction

#### Concentration calculation

Calculating the concentrations is highly dependent on the reaction in question and the spectra of the species involved. In this scenario, the integrals of the peaks corresponding to the educt and product are normalized by the integral of the triethylphosphate (TEP) peak, which serves as an internal standard. The resulting values are then multiplied by the known concentration of TEP to obtain the concentrations of educt and product(s).

In [ ]:
def process_integrals(fid_array: "FidArray") -> dict[str, list[float]]:
    """Extract species-specific deconvoluted integrals from the FID array."""
    ints = fid_array.deconvoluted_integrals.transpose()

    return {
        'chem01': ints[0].astype(float).tolist(),
        'chem02': ints[1].astype(float).tolist(),
        'chem04': ints[3].astype(float).tolist(),
        'chem03': ints[4].astype(float).tolist(),
    }


def calculate_concentrations(
    int_dict: dict[str, list[float]],
    standard_species: str = 'TEP',
    standard_concentration: float = 5.0,
) -> dict[str, list[float]]:
    """Normalize integrals to the internal standard and convert them to concentrations."""
    if standard_species not in int_dict:
        raise KeyError(f'Internal standard "{standard_species}" not found in int_dict.')

    standard_mean = float(np.mean(int_dict[standard_species]))
    if standard_mean == 0:
        raise ValueError('Mean integral of the internal standard is zero.')

    concentrations = {}
    for species_id, integrals in int_dict.items():
        concentrations[species_id] = (
            standard_concentration * np.asarray(integrals, dtype=float) / standard_mean
        ).tolist()

    return concentrations

In [ ]:
int_dict = process_integrals(p)
concentrations = calculate_concentrations(int_dict)
p.concentrations = concentrations

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
for k,v in concentrations.items():
    ax.plot(p.t, v, 's', mec='k', label=k)
ax.legend()
ax.set_xlabel('Time (min)')
ax.set_ylabel('Concentration (mM)')

#### Apply the concentrations to EnzymeML

The concentrations calculated in NMRpy can be applied back to the original or any other EnzymeML document by calling the `FidArray.apply_to_enzymeml()` method. If no EnzymeML document is provided to the method, the original document is used by default.

In [ ]:
enzymeml_doc = p.apply_to_enzymeml(measurement_id=CURRENT_MEASUREMENT)

print(enzymeml_doc.filter_measurements(id=CURRENT_MEASUREMENT)[0])

In [ ]:
pe.write_enzymeml(enzymeml_doc, path='./data/output/pgm-eno.json')

#### Save NMRPy data model

The NMRPy data model for the current measurement can be save as a JSON file, too. Thereby, all relevant NMR data and metadata are stored in a structured format.

In [ ]:
with open(f'./data/output/{p.enzymeml_document.filter_measurements(id=CURRENT_MEASUREMENT)[0].id}_data_model.json', 'w') as f:
    f.write(p.data_model.model_dump_json(indent=2))

---

### Disclosure

**Contributions**

If you wish to contribute to the EnyzmeML and/or NMRPy platforms, find us on our [EnzymeML GitHub](https://github.com/EnzymeML) and [NMRPy GitHub](https://github.com/NMRPy)!

**BSD 3-Clause License**

Copyright (c) 2026 Kamogelo Matenchi, Torsten Giess

Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:

1. Redistributions of source code must retain the above copyright notice, this list of conditions and the following disclaimer.

2. Redistributions in binary form must reproduce the above copyright notice, this list of conditions and the following disclaimer in the documentation and/or other materials provided with the distribution.

3. Neither the name of the copyright holder nor the names of its contributors may be used to endorse or promote products derived from this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS “AS IS” AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.